In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

# --- Caminhos do Dataset ---
DATASET_ROOT_FOLDER = '../../data/tiger2'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')
GT_TXT_PATH = os.path.join(DATASET_ROOT_FOLDER, 'tiger2_gt.txt')

In [ ]:
import itertools
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
from tracker.cluswisard_tracker import ClusWisardTracker

fixed_params = {
    'CLUSAZARD_BLEACHING_ACTIVATED': False,
    'CLUSAZARD_ACTIVATION_DEGREE': True,
    'CLUSAZARD_RETURN_CONFIDENCE': False,
    'CLUSAZARD_CLASSES_DEGREES': False,
}

param_grid = {
    'CLUSAZARD_ADDRESS_SIZE': [2],
    'CLUSAZARD_MIN_SCORE': [0.8],
    'CLUSAZARD_THRESHOLD': [2],
    'CLUSAZARD_DISCRIMINATOR_LIMIT': [1],
    'INPUT_PATTERN_SIDE': [32],
    'SEARCH_WINDOW_SCALE': [1.7, 2],
    'NUM_SEARCH_SAMPLES': [ 200, 500, 1000, 1500] 
}

combinations = list(itertools.product(*param_grid.values()))

def run_tracker(combo):
    params = fixed_params.copy()
    for k, v in zip(param_grid.keys(), combo):
        params[k] = v
    tracker = ClusWisardTracker(IMAGE_FOLDER, GT_TXT_PATH, params)
    outputs = tracker.track()
    max_per_frame = [obj['activationDegree'] for obj in outputs if 'activationDegree' in obj]
    max_activation = max(max_per_frame) if max_per_frame else 0.0
    return {
        'params': params.copy(),
        'max_activationDegree': max_activation
    }

# Paralelização com barra de progresso
results = []
with ProcessPoolExecutor() as executor:
    futures = [executor.submit(run_tracker, combo) for combo in combinations]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Executando Grid Search"):
        results.append(future.result())

# Ordenar por melhor resultado
results = sorted(results, key=lambda x: x['max_activationDegree'], reverse=True)

print("Melhor combinação:", results[0]['params'])
print("ActivationDegree máximo:", results[0]['max_activationDegree'])

# Plot
x_labels = [f"{i+1}" for i in range(len(results))]
y_values = [r['max_activationDegree'] for r in results]

plt.figure(figsize=(12, 6))
plt.plot(x_labels, y_values, marker='o')
plt.xticks(rotation=90)
plt.xlabel("Combinação de parâmetros")
plt.ylabel("ActivationDegree máximo por frame")
plt.title("Grid Search - ActivationDegree por combinação")
plt.tight_layout()
plt.show()
